# NR Completion — re-measuring Noise Resilience for the explain-func metrics

The June meta-evaluation's NR-stage metric calls for **max_sensitivity**,
**avg_sensitivity**, and **random_logit** failed silently, and a defect in the
NR estimator scored the all-NaN result as NR = 1.0 (fixed on master: failed
measurements now yield NaN; the artifact values are invalidated in
`results/meta_evaluation_reliability.csv` and those metrics' lambda is
currently AR-only). This notebook re-measures NR properly:

- same **seed-42 64-image sample** as the main meta-evaluation and the AR-v2 run,
- **5 noise seeds** per (model, FAE, metric) cell (the original NR protocol),
- per-unit Drive checkpointing (a unit = one (model, FAE, metric, seed) call),
- patches `nr_score` and `combined_reliability = (NR + AR)/2` for the
  affected rows, backing up the pre-NR CSV first.

**Estimated cost (T4):** ~6 h for max_sensitivity + random_logit
(`INCLUDE_AVG_SENSITIVITY = True` adds ~5 h). The occlusion x max_sensitivity
cells dominate. Resume-safe: re-run top to bottom after any disconnect.

**Prerequisites:** push master first (the clone asserts the NR fix is
present); GH_TOKEN Colab secret; T4 runtime; Drive assets as usual.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_THESIS = '/content/drive/MyDrive/thesis'
import os
os.makedirs(f'{DRIVE_THESIS}/results', exist_ok=True)
print('Drive mounted.')

## 2. Clone Repository (private — needs the GH_TOKEN Colab secret)

In [ ]:
import os
from google.colab import userdata
try:
    GH_TOKEN = userdata.get('GH_TOKEN')
    CLONE_URL = f'https://{GH_TOKEN}@github.com/dawkopagh/fae-metrics-master-thesis.git'
except Exception:
    print('No GH_TOKEN secret found - trying anonymous clone.')
    CLONE_URL = 'https://github.com/dawkopagh/fae-metrics-master-thesis.git'

THESIS_ROOT = '/content/thesis-repo'
CODE_DIR    = f'{THESIS_ROOT}/fae-metrics-master-thesis'
if os.path.exists(THESIS_ROOT):
    %cd {THESIS_ROOT}
    !git pull {CLONE_URL} master
else:
    !git clone {CLONE_URL} {THESIS_ROOT}
%cd {CODE_DIR}
print('HEAD commit:', end=' '); !git rev-parse HEAD

from pathlib import Path
src = Path('src/meta_evaluation/metaquantus_wrapper.py').read_text()
assert 'len(valid) < 2' in src and 'make_degraded_explain_func' in src, (
    'NR-fix commit missing - git push local master first, then re-run.')
print('NR fix present.')

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q quantus==0.6.0
import quantus, torch
print('quantus', quantus.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4. Weights from Drive (checksum-verified)

In [ ]:
RESNET_SHA     = '593dcb844b8359550e3d84667475bbd845f45a6cb388356480c0a55cb5173430'
SQUEEZENET_SHA = '8bbb43bbba4ee81e58e354295420bea33e55cfa2be11c31d82dce272d03b092d'
import os, hashlib
os.makedirs('weights', exist_ok=True)
!cp {DRIVE_THESIS}/weights/resnet18_isic2017.pth   weights/resnet18_isic2017.pth
!cp {DRIVE_THESIS}/weights/squeezenet_isic2017.pth weights/squeezenet_isic2017.pth
for path, expected in [('weights/resnet18_isic2017.pth', RESNET_SHA),
                       ('weights/squeezenet_isic2017.pth', SQUEEZENET_SHA)]:
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert actual == expected, f'Checksum mismatch for {path}'
    print(f'{path}: OK')

## 5. Test Split from Drive (fallback: re-download)

In [ ]:
import os, sys
sys.path.insert(0, '.')
_needs = False
for d in ['data/images/test', 'data/masks/test']:
    src = f'{DRIVE_THESIS}/{d}'
    if os.path.exists(src):
        os.makedirs(d, exist_ok=True)
        !cp -r {src}/* {d}/
        print(d, 'from Drive')
    else:
        _needs = True
if _needs:
    from src.data.download_isic import download_isic2017
    download_isic2017(dest_dir='data', skip_existing=True)
print('Data ready.')

## 6. Models, seed-42 sample, explainers, metrics

In [ ]:
import numpy as np, torch, quantus, pandas as pd
from src.models.classifiers import load_resnet18, load_squeezenet
from src.data.isic_dataset  import ISIC2017Dataset
from src.pipeline           import _make_explain_func
from src.meta_evaluation.metaquantus_wrapper import noise_resilience_test

SEED, N_SEEDS, META_N_IMAGES = 42, 5, 64
INCLUDE_AVG_SENSITIVITY = False   # not in M*; True adds ~5 h

device = 'cuda' if torch.cuda.is_available() else 'cpu'
models = {
    'resnet18':   load_resnet18('weights/resnet18_isic2017.pth',   device=device),
    'squeezenet': load_squeezenet('weights/squeezenet_isic2017.pth', device=device),
}
dataset = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
_rng = np.random.default_rng(SEED)
_sel = sorted(_rng.choice(len(dataset), size=META_N_IMAGES, replace=False).tolist())
samples = [dataset[i] for i in _sel]
images  = [s['image'] for s in samples]
imgs_np = [img.detach().cpu().numpy() for img in images]
_img_t = torch.stack(images).to(device)
with torch.no_grad():
    targets = models['resnet18'](_img_t).argmax(dim=1).tolist()
del _img_t
print(f'{len(images)} images (seed 42), targets ready.')

FAE_NAMES = ['integrated_gradients', 'saliency', 'gradcam',
             'deep_lift', 'guided_backprop', 'lrp', 'occlusion']
fae_funcs = {m: {n: _make_explain_func(n, models[m], m, device) for n in FAE_NAMES}
             for m in models}

metric_fns = {
    'max_sensitivity': quantus.MaxSensitivity(
        nr_samples=10, lower_bound=0.2,
        normalise=False, abs=False, return_aggregate=False, disable_warnings=True),
    'random_logit': quantus.RandomLogit(
        num_classes=3, abs=True, normalise=True,
        return_aggregate=False, disable_warnings=True),
}
if INCLUDE_AVG_SENSITIVITY:
    metric_fns['avg_sensitivity'] = quantus.AvgSensitivity(
        nr_samples=10, lower_bound=0.2,
        normalise=False, abs=False, return_aggregate=False, disable_warnings=True)
print(f'{len(models)} models x {len(FAE_NAMES)} FAE x {len(metric_fns)} metrics '
      f'= {len(models)*len(FAE_NAMES)*len(metric_fns)} NR cells (5 seeds each)')

## 7. Run NR — per-cell checkpointing, Drive-durable

Uses the repaired `noise_resilience_test` (failed measurements yield NaN,
never 1.0). Each finished cell appends to the Drive checkpoint immediately.

In [ ]:
import time, shutil
CKPT = f'{DRIVE_THESIS}/results/nr_completion_progress.csv'
done = {}
if os.path.exists(CKPT):
    ck = pd.read_csv(CKPT)
    done = {(r.model, r.fae_method, r.metric): float(r.nr_score) for r in ck.itertuples()}
    print(f'Resuming: {len(done)} cells done.')

results = dict(done)
work = [(m, f, met) for m in models for f in FAE_NAMES for met in metric_fns]
work = [w for w in work if w not in done]
print(f'{len(work)} cells to run.')

for i, (model_name, fae_name, metric_name) in enumerate(work, 1):
    model = models[model_name]
    explain_fn = fae_funcs[model_name][fae_name]
    t0 = time.perf_counter()
    attrs = explain_fn(model, np.stack(imgs_np), np.array(targets, dtype=np.int64))
    attributions = [np.asarray(attrs[j]) for j in range(len(imgs_np))]
    nr = noise_resilience_test(
        metric_fn=metric_fns[metric_name], model=model,
        images=imgs_np, attributions=attributions, targets=targets,
        explain_fn=explain_fn, n_seeds=N_SEEDS, device=device)
    elapsed = time.perf_counter() - t0
    key = (model_name, fae_name, metric_name)
    results[key] = nr['nr_score']
    row = pd.DataFrame([{'model': model_name, 'fae_method': fae_name,
                         'metric': metric_name, 'nr_score': nr['nr_score'],
                         'mean_score': nr.get('mean_score'),
                         'cv_score': nr.get('cv_score'),
                         'n_seeds': N_SEEDS, 'n_images': META_N_IMAGES,
                         'runtime_seconds': round(elapsed, 1)}])
    row.to_csv(CKPT, mode='a', header=not os.path.exists(CKPT), index=False)
    print(f'[{i:2d}/{len(work)}] {model_name:10s} {fae_name:20s} {metric_name:16s} '
          f'NR={nr["nr_score"] if nr["nr_score"]==nr["nr_score"] else float("nan"):.4f} {elapsed:.0f}s')
print('All NR cells complete.')

## 8. Patch the Reliability CSV and Save to Drive

Restores the NR leg for the re-measured rows; combined becomes
(NR + AR)/2 again. Pre-NR backup kept on Drive.

In [ ]:
REL_CSV = f'{THESIS_ROOT}/results/meta_evaluation_reliability.csv'
rel = pd.read_csv(REL_CSV)
backup = f'{DRIVE_THESIS}/results/meta_evaluation_reliability_pre_NR.csv'
if not os.path.exists(backup):
    shutil.copy2(REL_CSV, backup)
    print('Backup ->', backup)

patched = 0
for (model_name, fae_name, metric_name), nr_val in results.items():
    m = ((rel.model == model_name) & (rel.fae_method == fae_name)
         & (rel.metric == metric_name))
    assert m.sum() == 1
    idx = rel.index[m][0]
    ar = float(rel.at[idx, 'ar_score'])
    rel.at[idx, 'nr_score'] = nr_val
    rel.at[idx, 'combined_reliability'] = (
        0.5 * (nr_val + ar) if not (np.isnan(nr_val) or np.isnan(ar))
        else (nr_val if not np.isnan(nr_val) else (ar if not np.isnan(ar) else float('nan'))))
    patched += 1
print(f'Patched {patched} rows; valid combined:',
      rel.combined_reliability.notna().sum(), '/', len(rel))

out = f'{DRIVE_THESIS}/results/meta_evaluation_reliability.csv'
rel.to_csv('results/meta_evaluation_reliability_nr.csv', index=False)
shutil.copy2('results/meta_evaluation_reliability_nr.csv', out)
print('Saved ->', out)
print('''
NEXT STEPS (local): copy meta_evaluation_reliability.csv +
nr_completion_progress.csv from Drive into results/, then run
experiments/reconcile_ar_v2.py — it re-runs the chain and prints every
prose-cited number that moved, with the LaTeX checklist.''')